# 02 · 단면 제원 — 총단면과 환산단면

`ConcreteSection` 객체를 만들면 총단면 제원이 자동으로 계산된다.
원 문서의 `area_properties.ipynb` 에 대응한다.

In [ ]:
%matplotlib inline

import matplotlib.pyplot as plt
import numpy as np

# 한글 글꼴이 없는 환경에서도 그림이 깨지지 않도록 축 라벨은 ASCII 로 둔다
plt.rcParams["axes.unicode_minus"] = False
plt.rcParams["figure.dpi"] = 96

In [ ]:
from concreteproperties import ConcreteSection
from sectionproperties.pre.library import concrete_rectangular_section

from concreteproperties_kds import KDS


def beam_section(fck=27, fy=400):
    """400 x 600 보 단면 (상부 2-D16, 하부 4-D22, 피복 50 mm)."""
    kds = KDS(column_type="tie")
    conc = kds.create_concrete_material(compressive_strength=fck)
    steel = kds.create_steel_material(yield_strength=fy)

    geom = concrete_rectangular_section(
        d=600, b=400,
        dia_top=16, area_top=198.6, n_top=2, c_top=50,
        dia_bot=22, area_bot=387.1, n_bot=4, c_bot=50,
        n_circle=16, conc_mat=conc, steel_mat=steel,
    )
    conc_sec = ConcreteSection(geom)
    kds.assign_concrete_section(conc_sec)
    return kds, conc_sec


def column_section(fck=27, fy=400, column_type="tie"):
    """500 x 500 기둥 단면 (8-D22, 피복 50 mm)."""
    kds = KDS(column_type=column_type)
    conc = kds.create_concrete_material(compressive_strength=fck)
    steel = kds.create_steel_material(yield_strength=fy)

    geom = concrete_rectangular_section(
        d=500, b=500,
        dia_top=22, area_top=387.1, n_top=3, c_top=50,
        dia_bot=22, area_bot=387.1, n_bot=3, c_bot=50,
        dia_side=22, area_side=387.1, n_side=1, c_side=50,
        n_circle=16, conc_mat=conc, steel_mat=steel,
    )
    conc_sec = ConcreteSection(geom)
    kds.assign_concrete_section(conc_sec)
    return kds, conc_sec

In [ ]:
kds, conc_sec = beam_section()
conc_sec.plot_section()

## 총단면 제원

`GrossProperties` 의 단면2차모멘트는 탄성계수가 곱해진 **휨강성** ($EI$)
이다. 순수한 $I$ 가 필요하면 환산단면 제원을 쓴다.

In [ ]:
gross = kds.get_gross_properties()
gross.print_results()

## 콘크리트 환산단면 제원

In [ ]:
conc = conc_sec.concrete_geometries[0].material

transformed = kds.get_transformed_gross_properties(
    elastic_modulus=conc.elastic_modulus
)
transformed.print_results()

In [ ]:
print(f"환산 도심축 단면2차모멘트  Ixx_c = {transformed.ixx_c:,.0f} mm^4")
print(f"총단면 (400x600) 기준       bh^3/12 = {400 * 600 ** 3 / 12:,.0f} mm^4")
print(f"비                                  = "
      f"{transformed.ixx_c / (400 * 600 ** 3 / 12):.4f}")

철근이 환산되어 들어오므로 환산단면의 $I$ 가 콘크리트만의 $bh^3/12$ 보다
크다.